In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import random
import imageio
import os

# Set up output directory
output_dir = "mcts_frames"
os.makedirs(output_dir, exist_ok=True)

class Node:
    def __init__(self, name, parent=None):
        self.name = name
        self.parent = parent
        self.children = []
        self.visits = 0
        self.reward = 0.0

    def add_child(self, child):
        self.children.append(child)

# Initialize root node
root = Node("root")
nodes = {"root": root}
G = nx.DiGraph()
G.add_node("root")

def get_best_path(node):
    path = []
    while node:
        path.append(node.name)
        if node.children:
            node = max(node.children, key=lambda c: c.reward)
        else:
            break
    return path

# Simulation
frames = []
max_visits = 1
max_reward = 1

for step in range(300):
    current = random.choice(list(nodes.values()))
    current.visits += 1
    reward = random.random()
    current.reward += reward

    child_name = f"{current.name}_{len(current.children)}"
    child = Node(child_name, parent=current)
    child.reward = random.random() * (1 + step / 100)
    current.add_child(child)
    nodes[child_name] = child
    G.add_node(child_name)
    G.add_edge(current.name, child_name)

    max_visits = max(max_visits, current.visits, child.visits)
    max_reward = max(max_reward, current.reward, child.reward)

    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, seed=42)
    best_path = set(get_best_path(root))

    for node_name, node_obj in nodes.items():
        if node_obj.visits == 0:
            continue
        color = plt.cm.viridis(node_obj.reward / max_reward)
        alpha = min(1.0, node_obj.visits / max_visits)
        size = 300 if node_name in best_path else 150
        nx.draw_networkx_nodes(G, pos, nodelist=[node_name], node_color=[color],
                               node_size=size, alpha=alpha)
        nx.draw_networkx_labels(G, pos, labels={node_name: f"{node_obj.reward:.1f}"}, font_size=8)

    edge_colors, widths = [], []
    for u, v in G.edges():
        if u in best_path and v in best_path:
            edge_colors.append("red")
            widths.append(2.5)
        else:
            edge_colors.append("gray")
            widths.append(0.5)
    nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=widths)

    plt.title(f"Pipeline Execution Step {step + 1}")
    plt.axis("off")
    frame_path = os.path.join(output_dir, f"frame_{step:03d}.png")
    plt.savefig(frame_path)
    plt.close()
    frames.append(frame_path)

# Combine into GIF
with imageio.get_writer("pipeline_mcts.gif", mode='I', duration=0.15) as writer:
    for filename in frames:
        image = imageio.imread(filename)
        writer.append_data(image)

print("✅ Animated MCTS GIF saved as 'pipeline_mcts.gif'")


C:\Users\ernan\AppData\Local\Temp\ipykernel_17600\3857567283.py:94: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  image = imageio.imread(filename)


✅ Animated MCTS GIF saved as 'pipeline_mcts.gif'
